# A9 Report - Learning the Depth Information for Kinect Skeleton Avatars

## Assignment goal

The goal of A9 was to learn the depth information for Kinect skeleton avatars with deep learning. Following the Lab 9 lecture, the task was to predict the depth coordinate `z` for 13 Kinect joints from the corresponding 2D Kinect coordinates `x` and `y`, using short frame sequences as input. The lecture also required the full deep learning workflow to be documented: load data, define the network, compile it, use cross-validation, train it, evaluate it in a user-understandable metric, and save or use the trained model. In addition, we were expected to vary hyperparameters, explain the optimum setup, and describe dead ends.

## Data and problem formulation

The A9 experiments used 179 Kinect CSV sequence files containing a total of 24,005 frames. Each frame includes the joint positions for 13 body joints:

- head
- left shoulder
- left elbow
- right shoulder
- right elbow
- left hand
- right hand
- left hip
- right hip
- left knee
- right knee
- left foot
- right foot

For each frame, the input to the model was the 2D position of these joints, which gave 26 input features per frame (`13 joints x 2 coordinates`). The targets were the 13 corresponding depth values `z`.

To make the problem sequence-based, the data was converted into sliding windows of 20 frames. In the Dense and Conv1D experiments this produced sequence-to-sequence samples with input shape `(20, 26)` and target shape `(20, 13)`. In the recurrent LSTM/GRU experiments, the input shape was also `(20, 26)`, but the model predicted the 13 depth values for the last frame in the window.

To avoid leakage, the splits were done at the sequence level. Standardization was fitted on the training data only and then applied to validation and test data.

## Deep learning steps

### 1. Load Kinect movement data

The first step was to load the Kinect movement sequences from the CSV files and organize them by sequence identity. This made it possible to build sliding windows that preserved the temporal order within each movement. The sequence-level grouping was important because frames from the same movement must not appear in both training and testing data.

### 2. Define deep learning networks

Several model families were tested in the A9 folder:

- Dense models
- Conv1D models
- LSTM models
- GRU models

The Dense models flattened the 20-frame window and passed it through fully connected layers before predicting depth. The Conv1D models instead processed the temporal dimension explicitly with convolutions over time. The recurrent models used LSTM or GRU layers and predicted the depth values from the last hidden state.

This followed the lecture suggestion to compare multiple deep learning architectures for sequence data.

### 3. Compile the deep learning models

The models were trained as regression models. The main choices were:

- loss: MSE in the initial family comparison, and later Huber was also tested during Dense optimization
- optimizers: Adam in the initial Dense/Conv1D experiments, Adam and RMSprop for recurrent models, and AdamW during Dense optimization
- metric: MAE, reported in centimeters for interpretation

MAE in cm was the most important evaluation number because it tells how far the predicted depth is from the true depth in a way that is easy to understand.

### 4. Cross-validation

Cross-validation was used in all experiment groups, with sequence identity as the grouping variable:

- Dense and Conv1D comparison: 5-fold `GroupKFold`
- LSTM/GRU experiments: 18 sequence-level folds, roughly matching the lecture recommendation of holding out about 10 sequences per split
- Dense optimization: 5-fold grouped cross-validation

Within the training portion of each fold, a validation split was used for early stopping. This gave a more reliable estimate of generalization than a single train/test split.

### 5. Train the models

All model families were trained with early stopping so that training stopped when validation performance no longer improved. This limited overfitting and made the fold results more comparable.

The Dense/Conv1D family comparison showed that the better Dense models usually trained a little longer before reaching their best validation loss, while the weaker Conv1D variants often plateaued earlier.

### 6. Evaluate predictions

The main evaluation metric was mean absolute error in centimeters. This matched the lecture requirement to report an understandable accuracy measure. Per-joint MAE was also computed to see which joints were easiest and hardest to predict.

### 7. Use the trained model

The final Dense champion was packaged and saved together with its preprocessing objects and metadata. The saved artifacts in `ml/models/research/A9/reporting/results/dense_champion_20260415_053339` include:

- model weights
- input scaler
- target scaler
- training history
- configuration and metadata JSON files

This completed the final lecture step of actually using and saving the trained model.

## Comparing the model families

The initial Dense vs Conv1D comparison gave the following cross-validated results:

| Model variant | Mean test MAE (cm) | Std (cm) |
|---|---:|---:|
| Dense_256_128 | 3.576 | 0.454 |
| Conv1D_64_128_128_k5_dropout | 3.614 | 0.483 |
| Dense_512_256_128_dropout | 3.775 | 0.361 |
| Conv1D_64_128_k3 | 3.814 | 0.572 |

The best recurrent result from the LSTM/GRU notebook was:

| Model variant | Mean test MAE (cm) | Std (cm) |
|---|---:|---:|
| LSTM-1L-64-RMSprop | 3.896 | 0.762 |

The recurrent tuning runs stayed around the same level and did not beat the Dense family.

## Why the Dense model was chosen

The Dense model was chosen for three reasons.

First, it was the best-performing model family in the actual experiments. The initial Dense baseline achieved the lowest MAE among the Dense and Conv1D models, and even the best LSTM result remained worse than the Dense result.

Second, the problem appears to depend mostly on local pose geometry and short-term motion over a relatively short 20-frame window. For this kind of task, flattening the window and learning a direct nonlinear mapping can work very well. The results suggest that the task did not require the extra complexity of recurrent memory.

Third, the dataset is not very large in deep-learning terms. With 179 sequences, a simpler Dense architecture was easier to train, less sensitive than the recurrent models, and still expressive enough to capture the mapping from 2D pose to depth.

In short, Dense was chosen because it was both simpler and better.

## Dense model optimization

After Dense had been identified as the best family, the next step was to optimize it further. The optimization notebook tested both architecture changes and data-handling changes.

### Baseline for the optimization stage

The optimization stage used `Dense_baseline_256_128` as the reference point:

- mean test MAE: `3.731 cm`
- standard deviation: `0.572 cm`

### Variants that were tested

The main optimization variants were:

- `Dense_wider_512_256_128`
- `Dense_dropout_256_128_do02`
- `Dense_adamw_weightdecay`
- `Dense_huber_loss`
- `Dense_with_velocity`
- `Dense_target_scaled`
- `Dense_clip_targets_1_99`
- `Dense_stride2`
- `Dense_seq30`
- `Dense_best_guess_combo`

The best summary table from the saved result files was:

| Model variant | Mean test MAE (cm) | Std (cm) |
|---|---:|---:|
| Dense_best_guess_combo | 3.462 | 0.430 |
| Dense_target_scaled | 3.468 | 0.469 |
| Dense_clip_targets_1_99 | 3.568 | 0.403 |
| Dense_wider_512_256_128 | 3.575 | 0.455 |
| Dense_stride2 | 3.611 | 0.277 |
| Dense_adamw_weightdecay | 3.655 | 0.502 |
| Dense_baseline_256_128 | 3.731 | 0.572 |
| Dense_huber_loss | 3.800 | 0.555 |
| Dense_with_velocity | 3.812 | 0.564 |
| Dense_seq30 | 3.817 | 0.474 |
| Dense_dropout_256_128_do02 | 3.970 | 0.415 |

Compared with the optimization baseline, `Dense_best_guess_combo` improved the average error by about `0.268 cm`.

### What we changed and what helped

The optimization notebook also summarized which design choices helped on average:

- AdamW performed better than Adam
- Huber performed better than MSE on average
- adding velocity features helped on average
- scaling the targets gave the clearest single gain
- clipping targets gave a smaller but still useful improvement
- increasing the sequence length from 20 to 30 did not help

The strongest result came from combining several good ideas instead of changing only one hyperparameter.

### Final packaged Dense champion

The saved Dense champion was `Dense_best_guess_combo` with the following configuration:

- hidden layers: `(512, 256, 128)`
- activation: `GELU`
- dropout: `0.10`
- optimizer: `AdamW`
- learning rate: `0.001`
- weight decay: `1e-4`
- loss: `Huber`
- scheduler: enabled
- sequence length: `20`
- stride: `1`
- velocity features: enabled
- target scaling: enabled
- batch size: `32`

Its packaged validation result was:

- best validation loss: `0.0736`
- best epoch: `27`
- validation MAE: `2.888 cm`

Its cross-validated test result from the saved summary files was:

- mean test MAE: `3.462 cm`
- standard deviation: `0.430 cm`

This is about `3.17%` better than the initial Dense family champion at `3.576 cm`, about `4.19%` better than the best Conv1D result, and about `11.13%` better than the best recurrent result.

### Focused second-round tuning

A later focused sweep around the Dense champion tried stronger dropout, more weight decay, lower learning rates, and smaller architectures. The best observed result in that sweep was:

- `Dense_focus_dropout025`
- mean test MAE: `3.409 cm`
- standard deviation: `0.396 cm`

This shows that the Dense family could be pushed slightly further. However, the saved packaged model in the results folder remained `Dense_best_guess_combo`. The important conclusion did not change: Dense stayed clearly best.

## Per-joint performance

For the optimized Dense champion, the easiest joints were the hips, shoulders, and feet:

- left hip: `2.789 cm`
- right hip: `2.814 cm`
- left shoulder: `2.835 cm`
- left foot: `2.875 cm`
- right foot: `2.934 cm`
- right shoulder: `2.962 cm`

The hardest joints were the hands and elbows:

- left elbow: `3.881 cm`
- right elbow: `4.018 cm`
- left hand: `5.208 cm`
- right hand: `5.272 cm`

This pattern is reasonable. Hands move faster, have greater positional variation, and are harder to infer from 2D coordinates alone. The more stable joints near the torso are easier to predict.

## Dead ends and things that did not work well

Several ideas did not improve the model:

- the simpler Conv1D setup was weaker than Dense
- the larger Dense model with standard dropout did not beat the Dense baseline in the first family comparison
- increasing the sequence length to 30 frames made the Dense model worse
- a standard dropout version of the smaller Dense baseline hurt performance
- the recurrent models did not outperform Dense despite their extra temporal modeling capacity
- SGD was clearly the weakest recurrent optimizer

These dead ends are useful because they show that more complexity did not automatically produce better depth predictions.

## Conclusion

The A9 experiments show that the Kinect depth estimation task can be solved well with a relatively simple Dense neural network operating on short 20-frame windows of 2D joint positions. Among all tested model families, Dense gave the best overall trade-off between simplicity, trainability, and prediction quality.

The Dense family first won the architecture comparison against Conv1D, and it also remained better than the LSTM/GRU alternatives. After optimization, the packaged Dense champion reached about `3.46 cm` average depth error, and a later focused Dense sweep showed that the error could be pushed down to about `3.41 cm`. In practice, this means the model usually predicts Kinect joint depth within roughly three and a half centimeters.

The final conclusion is therefore that Dense was the right model choice for A9. The best improvements came from careful preprocessing and optimization, especially target scaling, better regularization, AdamW, and a stronger combined Dense architecture. The remaining weakness is hand prediction, which is still the hardest part of the task and would likely need richer features or more data to improve further.

## Note on the saved outputs

The A9 folder contains results from repeated notebook runs. The saved CSV and packaged model artifacts report `Dense_best_guess_combo` at `3.462 cm`, while a later focused notebook sweep reports a slightly lower Dense result of `3.409 cm`. This report uses both outputs transparently: the packaged champion is treated as the final saved model, and the focused sweep is treated as supporting evidence that the Dense family remained the best final choice.
